# Robinhood risk & behavior (tracker export)

Drawdown, calendar heatmap of daily P&L, hold-time vs outcome scatter, and return distribution. CSV import data only — not investment advice.

In [ ]:
bundle_path = ""
year = 0

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

nb_dir = Path.cwd()
if str(nb_dir) not in sys.path:
    sys.path.insert(0, str(nb_dir))

from lib.tracker_robinhood import (
    calendar_pnl_matrix,
    closed_trades_frame,
    daily_pnl_frame,
    drawdown_frame,
    equity_curve_frame,
    load_bundle,
    max_drawdown,
    pnl_distribution,
    win_rate_closed,
)

if not bundle_path:
    raise ValueError("Set bundle_path to a notebook-bundle JSON file.")

bundle = load_bundle(bundle_path)
display_year = year or bundle.get("year")
daily = daily_pnl_frame(bundle)
equity = equity_curve_frame(bundle)
closed = closed_trades_frame(bundle)
dd = drawdown_frame(equity)
mdd = max_drawdown(equity)
wr = win_rate_closed(closed)
print(f"Year {display_year} | trading days {len(daily)} | closed lots {len(closed)} | max drawdown ${mdd:,.2f}")
if wr is not None:
    print(f"Closed-lot win rate: {wr * 100:.1f}%")

In [ ]:
if dd.empty:
    print("No equity curve in bundle.")
else:
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(dd.index, dd["cumulativePnL"], color="#0f6d73", linewidth=1.5, label="Cumulative P&L")
    ax.fill_between(dd.index, dd["cumulativePnL"], dd["peak"], color="#dc2626", alpha=0.25, label="Drawdown")
    ax.axhline(0, color="#666", linewidth=0.8)
    ax.set_title(f"Equity curve & drawdown ({display_year})")
    ax.set_ylabel("USD")
    ax.legend(loc="upper left")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
cal = calendar_pnl_matrix(daily)
if cal.empty:
    print("Not enough daily P&L for calendar heatmap.")
else:
    fig, ax = plt.subplots(figsize=(10, max(3, len(cal) * 0.22)))
    sns.heatmap(
        cal,
        cmap="RdYlGn",
        center=0,
        annot=False,
        fmt=".0f",
        linewidths=0.5,
        cbar_kws={"label": "Daily realized P&L (USD)"},
        ax=ax,
    )
    ax.set_title(f"Daily P&L by ISO week & weekday ({display_year})")
    ax.set_xlabel("Weekday")
    ax.set_ylabel("ISO week")
    plt.tight_layout()
    plt.show()

In [ ]:
if closed.empty or "holdDays" not in closed.columns or "realizedPnL" not in closed.columns:
    print("No closed-trade legs in bundle.")
else:
    plot_df = closed.dropna(subset=["holdDays", "realizedPnL"]).copy()
    plot_df["win"] = plot_df["realizedPnL"] >= 0
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.scatterplot(
        data=plot_df,
        x="holdDays",
        y="realizedPnL",
        hue="win",
        palette={True: "#0d9488", False: "#dc2626"},
        alpha=0.75,
        ax=ax,
    )
    ax.axhline(0, color="#666", linewidth=0.8)
    ax.set_xlabel("Hold time (days)")
    ax.set_ylabel("Realized P&L (USD)")
    ax.set_title(f"Hold time vs outcome ({display_year})")
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(handles, ["Loss", "Win"], title="")
    plt.tight_layout()
    plt.show()

In [ ]:
dist = pnl_distribution(daily)
if dist.empty:
    print("No non-zero daily P&L days.")
else:
    fig, ax = plt.subplots(figsize=(9, 4))
    sns.histplot(dist, bins=min(30, max(8, len(dist) // 3)), kde=True, ax=ax, color="#0f6d73")
    ax.axvline(0, color="#666", linewidth=0.8)
    ax.set_title(f"Distribution of daily realized P&L ({display_year})")
    ax.set_xlabel("USD")
    plt.tight_layout()
    plt.show()